# Out-of-Distribution Detection: MSP vs KNN vs ViM

This notebook runs **three OOD detection methods on one shared model** and scores three
randomly drawn images — one from each dataset — with all three.

| | Dataset | Role |
|---|---|---|
| ID | CIFAR-10 test | what the model was trained to recognise |
| OOD 1 | CIFAR-100 test | **near-OOD** — natural 32×32 images, many classes semantically close to CIFAR-10 |
| OOD 2 | MNIST test | **far-OOD** — handwritten digits, grayscale |

**Model:** WideResNet-40-2 pretrained on CIFAR-10 (`wrn-40-2/cifar10/crossentropy`).

| Method | Score | Direction |
|---|---|---|
| **MSP** | `max(softmax(logits))` — the model's confidence | high = ID |
| **KNN** | distance to the K-th neighbour among 50,000 CIFAR-10 train features | high = OOD |
| **ViM** | virtual logit matching: residual outside the principal subspace, minus energy | high = OOD |

> **Note on direction:** MSP runs opposite to the other two. High MSP means *looks like ID*,
> while high KNN and high ViM mean *looks like OOD*. Keep this in mind when reading the results.

**Why all three belong in one notebook.** They share the same model and the same preprocessing,
and all three read the same 128-dimensional feature. Only under those conditions are the three
scores comparable against each other. Running them in three separate notebooks invites a silent
mismatch in preprocessing, which would invalidate any comparison.

**What "training" means here.** The model is not retrained. But KNN needs a *feature bank* built
from the 50,000 CIFAR-10 training images, and ViM needs to `fit` a principal subspace and an
alpha coefficient on those same images. This notebook makes **a single pass** over the training
set and shares the resulting features between both. MSP needs no fitting at all.

---

## Running this on Colab

1. **Enable the GPU:** `Runtime` → `Change runtime type` → `T4 GPU` → `Save`.
   Cell 2 prints `Device: cuda` when this is set correctly.
2. **Leave `USE_DRIVE = True`** in Cell 2. The datasets total ~335 MB, and Colab wipes its disk
   whenever it hands out a new runtime — without caching, every single session pays that
   download again.
3. `Runtime` → `Run all`.

The GPU does not make the download faster; these are two separate bottlenecks and each needs its
own fix. The final cell exports `ood_artifacts.npz`, which lets `OOD_Demo.ipynb` reproduce all
three methods in seconds without any dataset.

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

import subprocess
import sys

try:
    import pytorch_ood
except ModuleNotFoundError:
    # sys.executable installs into the exact Python running this notebook.
    # A bare "pip" may target a different interpreter, which produces the
    # confusing "No module named 'pytorch_ood'" error after a successful install.
    print("Installing pytorch-ood ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pytorch-ood"],
        check=True,
    )
    import pytorch_ood

print("Python      :", sys.version.split()[0])
print("pytorch-ood :", pytorch_ood.__version__)

## 1. Imports and configuration

### Parameters

- `USE_DRIVE` — cache the datasets in Google Drive under `MyDrive/ood_data`. Colab will ask for
  permission to access Drive. The first run still downloads ~335 MB; every run after that reads
  straight from Drive. Set it to `False` to keep everything on the local disk instead.
- `TRAIN_SUBSET` — `None` uses all 50,000 training images. A smaller number (say `10000`) makes
  a quick trial run, but **it changes the KNN and ViM scores** because the reference bank shrinks.
  Set it back to `None` for any result you intend to report.
- `SEED` — `None` draws a different set of three images on every run of Cell 9. Set an integer
  to reproduce one specific draw, which is useful when capturing a figure for a report.
- `K = 50` — number of neighbours for KNN.
- `VIM_D = 64` — dimensionality of the ViM principal subspace.

### Why the batch size depends on the device

On a GPU, a larger batch keeps more cores busy, so 256 is the better choice. On a CPU there is no
such wide parallelism, and a large batch mostly hurts cache locality — measured on a 20-core CPU,
batch 128 reached 486 images/sec while batch 256 dropped to 448. The cell picks whichever fits.

In [ ]:
# ============================================================
# CELL 2: IMPORTS + CONFIGURATION
# ============================================================

import os
import time

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10, CIFAR100, MNIST

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 256 if DEVICE.type == "cuda" else 128

MODEL_ID = "wrn-40-2/cifar10/crossentropy"

K     = 50      # KNN: distance to the 50th neighbour is the score
VIM_D = 64      # ViM: dimensionality of the principal subspace

# True  -> cache datasets in Google Drive, so they download only once, ever
# False -> re-download ~335 MB every time Colab hands out a new runtime
USE_DRIVE = True

# None  -> use all 50,000 CIFAR-10 training images (the numbers you should report)
# 10000 -> quick trial run; KNN and ViM scores will differ because the bank is smaller
TRAIN_SUBSET = None

# None = draw a different set of images on every run. Set an integer to reproduce one draw.
SEED = None

# ------------------------------------------------------------
# Where the datasets live
# ------------------------------------------------------------
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = "/content/drive/MyDrive/ood_data"
        print("Dataset cache:", DATA_ROOT)
    except ImportError:
        DATA_ROOT = "./data"
        print("Not running on Colab -> using local directory:", DATA_ROOT)
else:
    DATA_ROOT = "./data"
    print("DATA_ROOT:", DATA_ROOT, "(no cache -> Colab re-downloads every session)")

os.makedirs(DATA_ROOT, exist_ok=True)

# num_workers: Linux and Colab fork cheaply, so background workers pay off.
# Windows spawns fresh processes that re-import the whole notebook, which in
# Jupyter tends to hang and costs more than it saves.
NUM_WORKERS = 0 if os.name == "nt" else 2

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------
print("\nPyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("Device      :", DEVICE)
print("Batch size  :", BATCH_SIZE)
print("num_workers :", NUM_WORKERS)

if DEVICE.type == "cuda":
    # Inputs are a fixed 32x32, so the cuDNN autotuner has something to work with.
    torch.backends.cudnn.benchmark = True
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("CPU threads :", torch.get_num_threads(), f"across {os.cpu_count()} cores")
    print("\n" + "!" * 68)
    print("RUNNING ON CPU. The pass over the training set will be much slower.")
    print("On Colab: Runtime -> Change runtime type -> T4 GPU -> Save, then rerun.")
    print("Or set TRAIN_SUBSET = 10000 above for a quick trial.")
    print("!" * 68)

## 2. Load the pretrained model

`load_transform()` returns the exact preprocessing this checkpoint was trained with. Reusing that
object for every dataset is the safest way to keep preprocessing consistent — hand-typing `mean`
and `std` and getting one digit wrong would shift all three scores without any visible error.

This model exposes the three things the methods need:

- `model(x)` → 10 logits (MSP)
- `model.features(x)` → a 128-dimensional feature (KNN and ViM)
- `model.fc.weight` / `model.fc.bias` → the final layer (ViM)

In [ ]:
# ============================================================
# CELL 3: LOAD THE PRETRAINED WIDERESNET-40-2
# ============================================================

from pytorch_ood.model import load_model, load_transform

model = load_model(MODEL_ID).to(DEVICE)
model.eval()

# The checkpoint's own preprocessing -- reused for every dataset
base_transform = load_transform(MODEL_ID)

print("Model      :", type(model).__name__)
print("Device     :", next(model.parameters()).device)
print("Parameters :", f"{sum(p.numel() for p in model.parameters()):,}")
print("\nPreprocessing for 32x32 RGB images:")
print(base_transform)

## 3. Load the datasets

**CIFAR-10 train** is the reference data: KNN builds its feature bank from it and ViM fits on it.
The three test sets are where the demo images come from.

**MNIST needs extra handling.** Its images are 28×28 grayscale while the model expects 3×32×32:

1. `Resize((32, 32))` — match the input size
2. `Grayscale(num_output_channels=3)` — replicate the gray channel three times
3. then **append `base_transform` itself** — rather than retyping `Normalize`

The notebook also keeps an un-normalised copy of each test set (`*_raw`, `transform=None`) purely
for display. Normalised tensors passed to `imshow` would render with distorted colours.

In [ ]:
# ============================================================
# CELL 4: LOAD ALL FOUR DATASETS
# First run downloads ~335 MB. Later runs read from DATA_ROOT.
# ============================================================

_t0 = time.perf_counter()

_cached = os.path.isdir(os.path.join(DATA_ROOT, "cifar-10-batches-py"))
print("Datasets already present, skipping download."
      if _cached else "No cached data -> downloading ~335 MB. This happens only once.")

# MNIST: 28x28 grayscale -> 32x32 -> 3 channels -> the CIFAR-10 base_transform itself
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    base_transform,
])

# --- Reference data: CIFAR-10 TRAIN (KNN feature bank + ViM fit) ---
cifar10_train = CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=base_transform)

# --- The three test sets ---
cifar10_test  = CIFAR10(root=DATA_ROOT, train=False, download=True, transform=base_transform)
cifar100_test = CIFAR100(root=DATA_ROOT, train=False, download=True, transform=base_transform)
mnist_test    = MNIST(root=DATA_ROOT, train=False, download=True, transform=mnist_transform)

# --- Un-normalised copies, used only for displaying images ---
cifar10_raw  = CIFAR10(root=DATA_ROOT, train=False, download=False, transform=None)
cifar100_raw = CIFAR100(root=DATA_ROOT, train=False, download=False, transform=None)
mnist_raw    = MNIST(root=DATA_ROOT, train=False, download=False, transform=None)

print(f"\nFinished in {time.perf_counter() - _t0:.1f} s\n")

print("CIFAR-10 train :", len(cifar10_train), "(reference for KNN + ViM)")
print("CIFAR-10 test  :", len(cifar10_test),  "(ID)")
print("CIFAR-100 test :", len(cifar100_test), "(near-OOD)")
print("MNIST test     :", len(mnist_test),    "(far-OOD)")

print()
for name, ds in [("CIFAR-10", cifar10_test), ("CIFAR-100", cifar100_test), ("MNIST", mnist_test)]:
    img, _ = ds[0]
    assert tuple(img.shape) == (3, 32, 32), f"{name}: wrong transform, shape = {tuple(img.shape)}"
    print(f"{name:<10} shape after transform: {tuple(img.shape)}  OK")

## 4. One pass over CIFAR-10 train

This is the slowest step and the only one that touches the training set.

KNN needs all 50,000 features as its reference bank; ViM needs those same features to compute a
covariance, from which it derives the principal subspace and alpha. Because both read the **same
features from the same model**, the notebook extracts them **once** and shares the result, instead
of scanning the training set twice as two separate notebooks would.

MSP needs nothing from this step — it only reads the softmax of the logits.

In [ ]:
# ============================================================
# CELL 5: EXTRACT 128-D FEATURES FOR CIFAR-10 TRAIN
# Shared by KNN (feature bank) and ViM (fit)
# ============================================================

# TRAIN_SUBSET draws a random sample rather than taking the first N images,
# so that every class stays represented.
if TRAIN_SUBSET is None:
    fit_dataset = cifar10_train
    print(f"Using all {len(cifar10_train)} training images.")
else:
    g = torch.Generator().manual_seed(0)
    sel = torch.randperm(len(cifar10_train), generator=g)[:TRAIN_SUBSET]
    fit_dataset = Subset(cifar10_train, sel.tolist())
    print(f"TRIAL RUN: using only {TRAIN_SUBSET}/{len(cifar10_train)} training images.")
    print("-> KNN and ViM scores will differ from the real ones. Set TRAIN_SUBSET = None to report.")

train_loader = DataLoader(
    fit_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

feature_chunks, label_chunks = [], []

_t0 = time.perf_counter()

model.eval()
with torch.no_grad():
    for i, (images, labels) in enumerate(train_loader, start=1):
        feats = model.features(images.to(DEVICE, non_blocking=True))
        feature_chunks.append(feats.cpu())
        label_chunks.append(labels)

        if i % 40 == 0 or i == len(train_loader):
            rate = (i * BATCH_SIZE) / (time.perf_counter() - _t0)
            print(f"  batch {i}/{len(train_loader)}   {rate:,.0f} images/sec")

_elapsed = time.perf_counter() - _t0

feature_bank = torch.cat(feature_chunks)   # (N, 128)
label_bank   = torch.cat(label_chunks)     # (N,)

del feature_chunks, label_chunks

print("\n========== FEATURE BANK ==========")
print("Feature bank :", tuple(feature_bank.shape))
print("Label bank   :", tuple(label_bank.shape))
print("Memory       :", f"{feature_bank.numel() * 4 / 1e6:.1f} MB")
print("Elapsed      :", f"{_elapsed:.1f} s on {DEVICE.type.upper()}")

if TRAIN_SUBSET is not None:
    print("\n-> Trial run with a subset. Set TRAIN_SUBSET = None for reportable numbers.")

In [ ]:
# ============================================================
# CELL 6: MOVE THE FEATURE BANK TO THE DEVICE (for KNN)
# ============================================================

# cdist is much faster on a GPU. On CPU this is effectively a no-op.
try:
    feature_bank_dev = feature_bank.to(DEVICE)
    print("Feature bank on:", feature_bank_dev.device)
except RuntimeError as e:
    feature_bank_dev = feature_bank
    print("Not enough VRAM -> keeping the feature bank on the CPU.")
    print(" ", e)

In [ ]:
# ============================================================
# CELL 7: FIT ViM ON THE CIFAR-10 TRAIN FEATURES
# ============================================================

from pytorch_ood.detector import ViM

vim = ViM(
    model.features,
    d=VIM_D,
    w=model.fc.weight,
    b=model.fc.bias,
).to(DEVICE)

print("Fitting ViM ...")

# Reuse the features from Cell 5 instead of scanning the training set again.
# Older pytorch-ood releases lack fit_features(), so fall back to fit(loader).
if hasattr(vim, "fit_features"):
    vim.fit_features(feature_bank, label_bank)
    print("-> fitted from the existing features (fit_features).")
else:
    vim.fit(train_loader)
    print("-> this pytorch-ood release has no fit_features; refitted via fit(loader).")

print("\nViM fitted.")
print("Detector          :", vim)
print("Principal subspace:", tuple(vim.principal_subspace.shape))
print("alpha             :", float(vim.alpha))

## 5. Scoring one image with all three methods

A single function takes one preprocessed image and returns all three scores.

The important detail: it calls `model.features(x)` **once** and reuses that feature for all three:

- **MSP** — logits via `model.fc(feature)`, then `max(softmax(logits))`
- **KNN** — `cdist` from the feature to the bank, distance to the `K`-th neighbour
- **ViM** — `vim.predict_features(feature)`

This guarantees the three scores describe **the same representation** of the same image.

The function also returns the predicted CIFAR-10 class and the labels of the `K` nearest
neighbours, which help explain why a score came out the way it did.

In [ ]:
# ============================================================
# CELL 8: SCORE ONE IMAGE WITH ALL THREE METHODS
# ============================================================

CIFAR10_CLASSES = cifar10_raw.classes


@torch.no_grad()
def score_all_methods(image_tensor):
    """
    image_tensor: a preprocessed image of shape (3, 32, 32).

    Returns a dict with:
        msp       - max softmax probability          (high = ID)
        knn       - distance to the K-th neighbour   (high = OOD)
        vim       - ViM score                        (high = OOD)
        pred      - predicted CIFAR-10 class
        neighbors - labels of the K nearest neighbours
    """
    model.eval()

    x = image_tensor.unsqueeze(0).to(DEVICE)       # (1, 3, 32, 32)
    feat = model.features(x)                       # (1, 128) -- computed ONCE

    # --- MSP ---
    logits = model.fc(feat)                        # (1, 10)
    msp = F.softmax(logits, dim=1).max(dim=1).values.item()
    pred = logits.argmax(dim=1).item()

    # --- KNN ---
    q = feat.to(feature_bank_dev.device)
    dists = torch.cdist(q, feature_bank_dev).squeeze(0)
    knn_d, knn_i = torch.topk(dists, k=K, largest=False)

    # --- ViM, from that same feature ---
    vim_score = vim.predict_features(feat).item()

    return {
        "msp":       msp,
        "knn":       knn_d[-1].item(),
        "vim":       vim_score,
        "pred":      CIFAR10_CLASSES[pred],
        "neighbors": [CIFAR10_CLASSES[c] for c in label_bank[knn_i.cpu()].tolist()],
    }


# ------------------------------------------------------------
# Verify the shared-feature shortcut.
# WideResNet.forward() is exactly features() followed by fc(), so recomputing
# the logits from the feature must match model(x) bit for bit. If this assert
# ever fires, the MSP numbers above it are wrong.
# ------------------------------------------------------------
_x = cifar10_test[0][0].unsqueeze(0).to(DEVICE)
with torch.no_grad():
    assert torch.allclose(model.fc(model.features(_x)), model(_x), atol=1e-5), \
        "fc(features(x)) != model(x) -- the shared feature is not valid for MSP!"
print("model.fc(model.features(x)) == model(x)  -> sharing one feature is valid.\n")

print("Sanity check on image 0 of each dataset:\n")
print(f"{'Dataset':<18}{'MSP':>10}{'KNN':>12}{'ViM':>12}   predicted")
print("-" * 64)
for name, ds in [("CIFAR-10 (ID)", cifar10_test),
                 ("CIFAR-100 (OOD)", cifar100_test),
                 ("MNIST (OOD)", mnist_test)]:
    r = score_all_methods(ds[0][0])
    print(f"{name:<18}{r['msp']:>10.4f}{r['knn']:>12.4f}{r['vim']:>12.4f}   {r['pred']}")

## 6. Results — three random images

The cell below draws one random image from each dataset and scores all three methods.

**Rerun it as often as you like** to see different draws (`SEED = None`). The indices are printed,
so a draw worth keeping can be reproduced later.

In [ ]:
# ============================================================
# CELL 9: DRAW THREE RANDOM IMAGES AND SCORE THEM
# Rerun this cell for a different draw.
# ============================================================

rng = np.random.default_rng(SEED)

IDX_ID    = int(rng.integers(len(cifar10_test)))
IDX_C100  = int(rng.integers(len(cifar100_test)))
IDX_MNIST = int(rng.integers(len(mnist_test)))

# (display name, ID/OOD, preprocessed dataset, raw dataset for display, index)
panel = [
    ("CIFAR-10",  "ID",  cifar10_test,  cifar10_raw,  IDX_ID),
    ("CIFAR-100", "OOD", cifar100_test, cifar100_raw, IDX_C100),
    ("MNIST",     "OOD", mnist_test,    mnist_raw,    IDX_MNIST),
]

results = []
for name, kind, ds, raw_ds, idx in panel:
    r = score_all_methods(ds[idx][0])
    # .convert("RGB"): raw MNIST is a mode "L" (2-D) image; passing it straight
    # to imshow would colour it with the default colormap instead of grayscale.
    r.update(name=name, kind=kind, index=idx, image=raw_ds[idx][0].convert("RGB"))
    results.append(r)

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------
W = 20
print("=" * (12 + 3 * W))
print("THREE METHODS ON THREE RANDOM IMAGES")
print("=" * (12 + 3 * W))
print(f"{'':<12}" + "".join(f"{r['name'] + ' (' + r['kind'] + ')':>{W}}" for r in results))
print(f"{'index':<12}" + "".join(f"{'#' + str(r['index']):>{W}}" for r in results))
print("-" * (12 + 3 * W))
print(f"{'MSP':<12}" + "".join(f"{'Confidence: ' + format(r['msp'], '.2f'):>{W}}" for r in results))
print(f"{'KNN':<12}" + "".join(f"{'Distance: ' + format(r['knn'], '.2f'):>{W}}" for r in results))
print(f"{'ViM':<12}" + "".join(f"{'Score: ' + format(r['vim'], '.2f'):>{W}}" for r in results))
print("-" * (12 + 3 * W))
print(f"{'predicted':<12}" + "".join(f"{r['pred']:>{W}}" for r in results))
print("=" * (12 + 3 * W))

print("\nMSP high = looks ID    (max softmax probability)")
print(f"KNN high = looks OOD   (distance to the {K}th neighbour)")
print(f"ViM high = looks OOD   (virtual logit matching, d={VIM_D})")

print(f"\nClasses of the {K} nearest neighbours (first 10):")
for r in results:
    print(f"  {r['name']:<10} {', '.join(r['neighbors'][:10])}")

### Figure

The same data as the table above, laid out for a slide or a report.

In [ ]:
# ============================================================
# CELL 10: PLOT THE RESULTS
# ============================================================

BG    = "#1c1c1c"   # background
INK   = "#e8e8e6"   # primary text
MUTED = "#9a9a96"   # secondary text
LINE  = "#3a3a3a"   # rule
C_ID  = "#56b870"   # green - ID
C_OOD = "#e2724a"   # orange - OOD

MONO = "DejaVu Sans Mono"

fig = plt.figure(figsize=(11.5, 6.6), facecolor=BG)

# Three image panels on top; the left column is reserved for the row labels
gs = fig.add_gridspec(1, 3, left=0.22, right=0.97, top=0.86, bottom=0.50, wspace=0.26)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

for ax, r in zip(axes, results):
    color = C_ID if r["kind"] == "ID" else C_OOD

    ax.imshow(r["image"], interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_color(color)
        side.set_linewidth(1.6)

    ax.set_title(f"{r['kind']} IMAGE", fontsize=12.5, color=color,
                 fontfamily=MONO, pad=22)
    ax.text(0.5, 1.04, f"{r['name']}  ·  #{r['index']}",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=9, color=MUTED, fontfamily=MONO)

# Read the panel positions back so the numbers line up under their image
col_x = [ax.get_position().x0 for ax in axes]
col_w = axes[0].get_position().width

y = 0.385
for label, field_name, key in [("MSP", "Confidence:", "msp"),
                               ("KNN", "Distance:",   "knn"),
                               ("ViM", "Score:",      "vim")]:
    fig.text(0.05, y, label, fontsize=13.5, color=INK, fontfamily=MONO, va="center")

    for x, r in zip(col_x, results):
        color = C_ID if r["kind"] == "ID" else C_OOD
        fig.text(x, y, field_name, fontsize=11.5, color=MUTED,
                 fontfamily=MONO, va="center", ha="left")
        fig.text(x + col_w, y, f"{r[key]:.2f}", fontsize=11.5, color=color,
                 fontfamily=MONO, va="center", ha="right")

    y -= 0.088

fig.add_artist(plt.Line2D([0.04, 0.97], [0.45, 0.45], color=LINE, linewidth=1.0))

fig.text(0.04, 0.085,
         "MSP high = looks ID        KNN high = looks OOD        ViM high = looks OOD",
         fontsize=10, color=MUTED, fontfamily=MONO)
fig.text(0.04, 0.035,
         f"WideResNet-40-2 / CIFAR-10   ·   KNN K={K}   ·   ViM d={VIM_D}",
         fontsize=10, color=MUTED, fontfamily=MONO)

plt.show()

## 7. Reading the results

All three scores come from **the same 128-dimensional feature** of the same image; only the
interpretation differs.

**MSP** looks only at the softmax of the final layer. Softmax normalises over exactly the 10
CIFAR-10 classes, so it has no way to express *"none of these"* — OOD images routinely receive
very high confidence. An MNIST digit scoring 0.99 is not a bug; it is the overconfidence problem
that motivated every method that came after MSP.

**KNN** ignores the classifier head and asks a different question: among the 50,000 training
images, does this one land in a crowded or a sparse region of feature space? The neighbour list
printed in Cell 9 makes this visible — ID images tend to draw neighbours from a single class,
while OOD images draw a scattered mix.

**ViM** combines both signals: the part of the feature lying **outside** the principal subspace of
the training data (unfamiliar images have a large residual), minus the energy of the logits. That
is why ViM usually beats MSP on near-OOD, where softmax has already saturated.

**A single random draw proves nothing.** Three images only illustrate how the methods score
individual inputs. Any given draw can show a CIFAR-100 image that all three rate as ID-looking —
near-OOD is genuinely hard, and the per-image scores overlap substantially between datasets.
Deciding which method is better requires all 10,000 images per set and metrics such as AUROC and
FPR95. Rerun Cell 9 and Cell 10 a few times to see how much the numbers move between draws.

## 8. Saving the figure and the images

Writes everything from the draw above into `results/`:

| File | Contents |
|---|---|
| `result_three_methods.png` | the figure from Cell 10, at 200 DPI |
| `<ID/OOD>_<dataset>_<index>_32x32.png` | the three images at native size — what the model saw |
| `<ID/OOD>_<dataset>_<index>_x8.png` | the same images at 8× for slides |
| `scores.csv` | MSP / KNN / ViM for the three images |
| `scores.json` | the same, plus the 50 nearest-neighbour labels |

The upscaled copies use `NEAREST` rather than `BILINEAR`: smooth interpolation of a 32×32 image
looks blurry and invents detail, while nearest-neighbour keeps the pixels square and honest.

`fig.savefig()` still works after `plt.show()` — showing a figure renders it, it does not destroy
the `Figure` object.

In [ ]:
# ============================================================
# CELL 11: SAVE THE FIGURE, THE THREE IMAGES, AND THE SCORES
# Requires: results (Cell 9) and fig (Cell 10)
# ============================================================

import json
import shutil

from PIL import Image

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

saved = []

# --- 1. The figure ---
panel_path = os.path.join(OUT_DIR, "result_three_methods.png")
fig.savefig(panel_path, dpi=200, facecolor=fig.get_facecolor(), bbox_inches="tight")
saved.append(panel_path)

# --- 2. The three images, at native size and at 8x ---
for r in results:
    stem = f"{r['kind']}_{r['name']}_{r['index']}"
    img = r["image"].convert("RGB")

    p_native = os.path.join(OUT_DIR, f"{stem}_32x32.png")
    img.save(p_native)
    saved.append(p_native)

    p_big = os.path.join(OUT_DIR, f"{stem}_x8.png")
    img.resize((img.width * 8, img.height * 8), Image.NEAREST).save(p_big)
    saved.append(p_big)

# --- 3. The scores ---
csv_path = os.path.join(OUT_DIR, "scores.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("dataset,kind,index,MSP,KNN,ViM,predicted\n")
    for r in results:
        f.write(f"{r['name']},{r['kind']},{r['index']},"
                f"{r['msp']:.6f},{r['knn']:.6f},{r['vim']:.6f},{r['pred']}\n")
saved.append(csv_path)

json_path = os.path.join(OUT_DIR, "scores.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        [{k: r[k] for k in
          ("name", "kind", "index", "msp", "knn", "vim", "pred", "neighbors")}
         for r in results],
        f, ensure_ascii=False, indent=2,
    )
saved.append(json_path)

print(f"Saved {len(saved)} files to {os.path.abspath(OUT_DIR)}\n")
for p in saved:
    print(f"  {os.path.getsize(p):>9,} bytes   {os.path.basename(p)}")

# --- 4. On Colab, zip and download in one go ---
zip_path = shutil.make_archive("ood_results", "zip", OUT_DIR)
print(f"\nArchived: {zip_path}  ({os.path.getsize(zip_path):,} bytes)")

try:
    from google.colab import files
    files.download(zip_path)
    print("Your browser will prompt for the download.")
    print("If the popup is blocked, allow it and rerun this cell.")
except ImportError:
    print("Not on Colab -> the file is at:", os.path.abspath(zip_path))

## 9. Exporting `ood_artifacts.npz`

The final cell packages everything `OOD_Demo.ipynb` needs to reproduce all three methods **without
downloading a single dataset and without touching the training set**. The demo then runs in
seconds instead of minutes.

What is worth noticing is how small the "trained" state of all three methods actually is:

| Component | Used by | Size |
|---|---|---|
| `feature_bank`, 50,000 × 128 | KNN | ~25 MB |
| `label_bank`, 50,000 | KNN (neighbour classes) | 50 KB |
| `principal_subspace` 128 × 64, plus `alpha` | ViM | 33 KB |
| — | MSP | **nothing** — the model is enough |
| 200 images per dataset | drawing demo images | ~1.4 MB |

MSP costs nothing because it only reads a softmax. ViM costs almost nothing because all it learned
is one 128 × 64 projection and a single scalar. The entire footprint is KNN's feature bank, and
that is inherent to the method: KNN is non-parametric, so it **must** carry its reference data.

**Provenance.** The export records a SHA-256 digest of the feature bank, the number of training
images behind it, the model identifier, and a timestamp. The digest is printed here and printed
again by the demo notebook when it loads the file, so the two can be compared directly to confirm
the demo is running on the feature bank produced by this training run.

In [ ]:
# ============================================================
# CELL 12: EXPORT ood_artifacts.npz FOR THE DEMO NOTEBOOK
# Requires: feature_bank, label_bank (Cell 5), vim (Cell 7), *_raw (Cell 4)
# ============================================================

import hashlib

from datetime import datetime, timezone

ARTIFACT_PATH = "ood_artifacts.npz"

N_POOL    = 200     # images per dataset carried along for the demo
POOL_SEED = 12345   # fixed, so the bundled pool is reproducible

# float32 preserves the numbers exactly (~25 MB). np.float16 halves the file
# at roughly 0.05% error -- effectively invisible in the KNN score, but then
# the demo no longer matches this notebook exactly.
FEATURE_DTYPE = np.float32

_rng_pool = np.random.default_rng(POOL_SEED)


def take_pool(raw_ds, n):
    """Sample n images from a raw (un-normalised) dataset as uint8 arrays."""
    idx = _rng_pool.choice(len(raw_ds), size=n, replace=False)
    imgs = np.stack([np.asarray(raw_ds[int(i)][0]) for i in idx])
    labels = np.array([raw_ds[int(i)][1] for i in idx], dtype=np.int16)
    return idx.astype(np.int32), imgs, labels


idx10,  img10,  lab10  = take_pool(cifar10_raw,  N_POOL)   # (N, 32, 32, 3) uint8
idx100, img100, lab100 = take_pool(cifar100_raw, N_POOL)   # (N, 32, 32, 3) uint8
idxmn,  imgmn,  labmn  = take_pool(mnist_raw,    N_POOL)   # (N, 28, 28)    uint8

print("Demo image pools:")
print("  CIFAR-10 :", img10.shape, img10.dtype)
print("  CIFAR-100:", img100.shape, img100.dtype)
print("  MNIST    :", imgmn.shape, imgmn.dtype)

# ------------------------------------------------------------
# Provenance digest, computed over the exact bytes being written
# ------------------------------------------------------------
fb_array = feature_bank.cpu().numpy().astype(FEATURE_DTYPE)
fb_sha256 = hashlib.sha256(fb_array.tobytes()).hexdigest()

# ------------------------------------------------------------
# Write the archive.
# vim.u is not strictly needed -- ViM.__init__ recomputes it from w and b --
# but storing it lets the demo verify it loaded the very same model.
# ------------------------------------------------------------
np.savez_compressed(
    ARTIFACT_PATH,

    # --- KNN ---
    feature_bank=fb_array,
    label_bank=label_bank.cpu().numpy().astype(np.int8),

    # --- ViM ---
    vim_principal_subspace=vim.principal_subspace.cpu().numpy().astype(np.float32),
    vim_alpha=np.float32(vim.alpha),
    vim_u=vim.u.cpu().numpy().astype(np.float32),

    # --- Demo image pools ---
    pool_cifar10=img10,    pool_cifar10_labels=lab10,    pool_cifar10_idx=idx10,
    pool_cifar100=img100,  pool_cifar100_labels=lab100,  pool_cifar100_idx=idx100,
    pool_mnist=imgmn,      pool_mnist_labels=labmn,      pool_mnist_idx=idxmn,

    # --- Provenance / configuration ---
    model_id=np.array(MODEL_ID),
    knn_k=np.int32(K),
    vim_d=np.int32(VIM_D),
    n_train=np.int32(len(feature_bank)),
    feature_sha256=np.array(fb_sha256),
    torch_version=np.array(torch.__version__),
    device_used=np.array(DEVICE.type),
    cifar10_classes=np.array(cifar10_raw.classes),
    created_at=np.array(datetime.now(timezone.utc).isoformat(timespec="seconds")),
)

size_mb = os.path.getsize(ARTIFACT_PATH) / 1e6
print(f"\nWrote {ARTIFACT_PATH}  ({size_mb:.1f} MB)")

# ------------------------------------------------------------
# Read it back and verify nothing was corrupted on the way out
# ------------------------------------------------------------
_chk = np.load(ARTIFACT_PATH, allow_pickle=False)

print("\nArchive contents:")
for key in _chk.files:
    arr = _chk[key]
    shape = str(arr.shape) if arr.shape else "(scalar)"
    print(f"  {key:<24} {shape:<18} {arr.dtype}")

assert np.array_equal(_chk["feature_bank"], fb_array), "feature_bank was not written correctly!"
assert hashlib.sha256(_chk["feature_bank"].tobytes()).hexdigest() == fb_sha256, "digest mismatch!"
print("\n-> Read back and verified against the data in memory.")

print("\n" + "=" * 72)
print("PROVENANCE -- compare these against what OOD_Demo.ipynb prints")
print("=" * 72)
print(f"  model              : {MODEL_ID}")
print(f"  training images    : {len(feature_bank):,}")
print(f"  feature bank shape : {tuple(fb_array.shape)}  {fb_array.dtype}")
print(f"  KNN K              : {K}")
print(f"  ViM d              : {VIM_D}")
print(f"  ViM alpha          : {float(vim.alpha):.6f}")
print(f"  device used        : {DEVICE.type}")
print(f"  created at (UTC)   : {str(_chk['created_at'])}")
print(f"  feature SHA-256    : {fb_sha256}")
print("=" * 72)

# ------------------------------------------------------------
# On Colab, download the archive
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(ARTIFACT_PATH)
    print("\nYour browser will prompt to download ood_artifacts.npz.")
    print("If the popup is blocked, allow it and rerun this cell.")
except ImportError:
    print("\nFile is at:", os.path.abspath(ARTIFACT_PATH))